# BSM — CORRECTED RUN — comparison of the four architectures

Loads `results_corr/metrics_*_corrected.json` (written by notebooks 05-08 corrected)
and the paper-faithful `results/metrics_*.json` (notebooks 05-08 faithful) and prints
a single comparison table.

Paper Table 9 numbers are printed **as a reference only** — they are the paper's own
values, never mixed with the measured values of this repository.


In [ ]:
# --- corrected-run comparison -------------------------------------------------------
import json, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PROJECT = Path("/content/drive/MyDrive/bsm_paper_replication")
RES = PROJECT / "results_corr"
MODELS = [("dresnet50", "D-ResNet-50"), ("resnet18", "ResNet-18"), ("resnet34", "ResNet-34"), ("vgg16", "VGG16")]
CLASSES = ["Ilmenite", "Rutile", "Zircon", "Garnet", "Monazite", "Sillimanite"]

rows = []
cmats = {}
for key, pretty in MODELS:
    p = RES / f"metrics_{key}_corrected.json"
    if not p.exists():
        print("MISSING:", p.name)
        continue
    m = json.load(open(p))
    ts = m["test_split"]
    rows.append({"Model": pretty, "Type": "CORRECTED", "Best epoch": m.get("best_epoch", -1),
                 "Accuracy": ts["accuracy"], "Macro P": ts["macro_precision"], "Macro R": ts["macro_recall"],
                 "Macro F1": ts["macro_f1"], "Macro Spec": ts["macro_specificity"], "n test": ts["n_test"]})
    cmats[pretty] = np.array(ts["confusion_matrix"])

# faithful (paper-faithful) results for side-by-side context, if present
for key, pretty in MODELS:
    p = PROJECT / "results" / f"metrics_{key}.json"
    if not p.exists():
        continue
    m = json.load(open(p))
    ts = m.get("test_split", m)
    rows.append({"Model": pretty, "Type": "FAITHFUL", "Best epoch": m.get("best_epoch", -1),
                 "Accuracy": ts["accuracy"], "Macro P": ts["macro_precision"], "Macro R": ts["macro_recall"],
                 "Macro F1": ts["macro_f1"], "Macro Spec": ts["macro_specificity"], "n test": ts.get("n_test", 0)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\nNOTE: paper Table 9 values (for reference only, NOT measured here):")
print("  VGG16 .83/.89/.83 | ResNet18 .78/.87/.78 | ResNet34 .83/.89/.83 | D-ResNet .89/.90/.89 (acc/prec/recall)")

out = {"corrected": df[df["Type"] == "CORRECTED"].to_dict("records"),
       "faithful": df[df["Type"] == "FAITHFUL"].to_dict("records"),
       "paper_table9_reference": {"vgg16": [0.83, 0.89, 0.83], "resnet18": [0.78, 0.87, 0.78],
                                  "resnet34": [0.83, 0.89, 0.83], "dresnet50": [0.89, 0.90, 0.89]}}
with open(RES / "comparison_corr.json", "w") as f:
    json.dump(out, f, indent=2)

if cmats:
    n = len(cmats)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.6))
    if n == 1:
        axes = [axes]
    for ax, (pretty, cm) in zip(axes, cmats.items()):
        im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(6), CLASSES, rotation=45, ha="right", fontsize=7)
        ax.set_yticks(range(6), CLASSES, fontsize=7)
        ax.set_title(f"{pretty} (corrected)", fontsize=9)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=6,
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
    fig.colorbar(im, ax=axes, fraction=0.02)
    fig.tight_layout()
    fig.savefig(RES / "confusion_matrices_corr.png", dpi=150)
    print("saved figure", RES / "confusion_matrices_corr.png")

print("COMPARISON DONE")


In [ ]:
# --- persist the executed notebook ---------------------------------------------------
import glob, shutil, os
os.makedirs(os.path.join(PROJECT, "notebooks"), exist_ok=True)
cands = glob.glob("/content/drive/MyDrive/Colab Notebooks/*.ipynb")
if cands:
    newest = max(cands, key=os.path.getmtime)
    dst = os.path.join(PROJECT, "notebooks", "_autosave_09_corr_comparison.ipynb")
    shutil.copy(newest, dst)  # staging only; the executed notebook is copied in afterwards
    print("copied", newest, "->", dst)
else:
    print("no autosaved notebook found")
